# 00 — Setup do Ambiente OCI

Notebook de configuracao inicial do ambiente OCI Data Science para o pipeline de risco de credito.

**Objetivo**: Validar dependencias, autenticacao OCI, conectividade com Object Storage e versoes das bibliotecas.

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
# Instalacao de dependencias a partir do requirements.txt
!pip install -r ../requirements.txt -q

In [ ]:
# Configuracao de threading para evitar contenção de CPU em ambientes multi-core OCI
import os

NCPUS = str(os.cpu_count() or 4)
os.environ["OMP_NUM_THREADS"] = NCPUS
os.environ["OPENBLAS_NUM_THREADS"] = NCPUS
os.environ["MKL_NUM_THREADS"] = NCPUS

# Patch de compatibilidade sklearn (versoes >= 1.3)
import sklearn
sklearn.set_config(transform_output="default")

print(f"Threading configurado: {NCPUS} CPUs")

In [ ]:
# Importar configuracoes centralizadas do projeto
import sys
sys.path.insert(0, "..")

from config import *
from utils import *

print(f"NAMESPACE: {NAMESPACE}")
print(f"BUCKETS: {BUCKETS}")
print(f"SAFRAS: {SAFRAS}")
print(f"TARGET: {TARGET}")
print(f"SELECTED_FEATURES: {len(SELECTED_FEATURES)} features")

In [ ]:
# Autenticacao OCI — tenta Resource Principal (Data Science), fallback para config file
import oci

try:
    rps = oci.auth.signers.get_resource_principals_signer()
    config_oci = {}
    signer = rps
    auth_method = "Resource Principal"
    print("[OK] Autenticacao via Resource Principal (OCI Data Science)")
except Exception:
    config_oci = oci.config.from_file()
    signer = None
    auth_method = "Config File (~/.oci/config)"
    print(f"[OK] Autenticacao via Config File — Tenancy: {config_oci.get('tenancy', 'N/A')}")

print(f"Metodo de autenticacao: {auth_method}")

In [ ]:
# Validar conectividade com Object Storage — listar buckets
if signer:
    os_client = oci.object_storage.ObjectStorageClient(config={}, signer=signer)
else:
    os_client = oci.object_storage.ObjectStorageClient(config_oci)

namespace_name = os_client.get_namespace().data
print(f"Namespace OCI: {namespace_name}")

# Listar buckets do compartment (necessita compartment_id)
compartment_id = config_oci.get("tenancy", os.environ.get("NB_SESSION_COMPARTMENT_OCID", ""))
if compartment_id:
    buckets_response = os_client.list_buckets(namespace_name, compartment_id)
    print(f"\nBuckets encontrados ({len(buckets_response.data)}):")
    for b in buckets_response.data:
        marker = " <-- PROJETO" if "pod-academy" in b.name else ""
        print(f"  - {b.name}{marker}")
else:
    print("[WARN] Compartment ID nao disponivel — validacao de buckets ignorada")

print("\n[OK] Conectividade com Object Storage validada")

In [ ]:
# Versoes das bibliotecas do pipeline
import platform
import numpy as np
import pandas as pd
import sklearn
import lightgbm
import xgboost
import catboost

try:
    import pyspark
    pyspark_version = pyspark.__version__
except ImportError:
    pyspark_version = "N/A (nao instalado localmente)"

versions = {
    "Python": platform.python_version(),
    "PySpark": pyspark_version,
    "scikit-learn": sklearn.__version__,
    "LightGBM": lightgbm.__version__,
    "XGBoost": xgboost.__version__,
    "CatBoost": catboost.__version__,
    "NumPy": np.__version__,
    "pandas": pd.__version__,
}

print("Versoes do Ambiente:")
print("-" * 40)
for lib, ver in versions.items():
    print(f"  {lib:15s}: {ver}")
print("-" * 40)

## Validacao Completa

Se todas as celulas acima executaram sem erro, o ambiente esta pronto:

- **Dependencias**: Instaladas via `requirements.txt`
- **Threading**: Configurado para CPUs disponiveis (OMP, OpenBLAS, MKL)
- **Config**: `config.py` e `utils.py` importados com sucesso
- **Autenticacao OCI**: Resource Principal ou Config File validado
- **Object Storage**: Conectividade confirmada, buckets listados
- **Bibliotecas**: Versoes compativeis com o pipeline

Proximo passo: `01_ingest_landing_bronze.ipynb`